# Solving a Power Flow

> **Set up**
>
> To run this notebook, first install the Julia kernel for Jupyter Notebooks using [IJulia](https://julialang.github.io/IJulia.jl/stable/manual/installation/), then [create an environment](https://pkgdocs.julialang.org/v1/environments/) for this tutorial with the packages listed with `using <PackageName>` further down.
>
> This tutorial has demonstrated compatibility with these package versions. If you run into any errors, first check your package versions for consistency using `Pkg.status()`.
>
 > ```
 > Status `~/work/PowerFlows.jl/PowerFlows.jl/docs/Project.toml`
 >   [a93c6f00] DataFrames v1.8.2
 >   [864edb3b] DataStructures v0.19.4
 >   [ffbed154] DocStringExtensions v0.9.5
 >   [e30172f5] Documenter v1.17.0
 >   [d12716ef] DocumenterInterLinks v1.1.0
 >   [35a29f4d] DocumenterTools v0.1.21
 >   [98b081ad] Literate v2.21.0
 >   [91a5bcdd] Plots v1.41.6
 >   [94fada2c] PowerFlows v0.16.3 `~/work/PowerFlows.jl/PowerFlows.jl`
 >   [f00506e0] PowerSystemCaseBuilder v2.2.1
 >   [bcd98974] PowerSystems v5.8.0
 >   [08abe8d2] PrettyTables v3.3.2
 > 
 > ```

In this tutorial, you'll solve power flows on a 5-bus test system using three different
solvers and compare their results.

## Building a System
To get started, load the needed packages. We're using a standard test system and want to
keep output clean, so we adjust the logging settings to filter out a few precautionary warnings.

In [ ]:
using PowerSystemCaseBuilder
using PowerFlows
using PowerSystems
using Logging

disable_logging(Logging.Warn)

Create a `System` from [PowerSystemCaseBuilder.jl](https://github.com/NREL-Sienna/PowerSystemCaseBuilder.jl):

In [ ]:
sys = build_system(MatpowerTestSystems, "matpower_case5_sys")

## DC Power Flow
`DCPowerFlow` solves for bus voltage angles using the bus admittance matrix,
then computes branch flows from the angle differences. Create a `DCPowerFlow` solver:

In [ ]:
pf_dc = DCPowerFlow()

Solve the power flow with `solve_power_flow`. For DC methods, pass a
`FlowReporting` mode so flows are reported on the
same arc basis as elsewhere in the package:

In [ ]:
dc_results = solve_power_flow(pf_dc, sys, FlowReporting.ARC_FLOWS)

The result is a `Dict{String, Dict{String, DataFrame}}`. The outer key is the time step
name: `"1"`. The inner dictionary stores the power flow results at that time step:
`"bus_results"` for bus data and `"flow_results"` for AC line data.
(There is also a third key, `"lcc_results"`, for HVDC lines, but this system
contains no such components, so the matching dataframe will be empty.) Inspect `"bus_results"`:

In [ ]:
dc_results["1"]["bus_results"]

Notice that `Vm` (voltage magnitude) is 1.0 for all buses, and `Q_gen` and `Q_load` are 0.
This is expected for DC power flow, which assumes flat voltage magnitudes and ignores reactive power.

In [ ]:
dc_results["1"]["flow_results"]

Likewise, `Q_from_to` and `Q_to_from` (reactive power flow on the line) are zero, for all lines.

## PTDF DC Power Flow
`PTDFDCPowerFlow` computes branch flows directly from bus power injections using
the Power Transfer Distribution Factor matrix, without solving for voltage angles as an
intermediate step. (This means we can omit the angle computation in contexts where we only
care about line flows, though we don't have that option implemented here.) Create a `PTDFDCPowerFlow`
solver:

In [ ]:
pf_ptdf = PTDFDCPowerFlow()

As before, solve the power flow with `solve_power_flow`:

In [ ]:
ptdf_results = solve_power_flow(pf_ptdf, sys, FlowReporting.ARC_FLOWS)

Look at the bus results:

In [ ]:
ptdf_results["1"]["bus_results"]

The results match `DCPowerFlow`, as they should: the two are mathematically equivalent.
For very large systems where forming the full PTDF matrix would be too expensive,
consider `vPTDFDCPowerFlow`, which computes the same results without
storing the dense matrix.

## AC Power Flow
Create an `ACPowerFlow` solver:

In [ ]:
pf_ac = ACPowerFlow()

Solve the power flow:

In [ ]:
ac_results = solve_power_flow(pf_ac, sys)

AC results are returned as a flat `Dict{String, DataFrame}`, with the same keys as
before: `"bus_results"`, `"flow_results"` (AC lines), and `"lcc_results"` (HVDC lines).
(We don't support multi-period AC power flows yet.) Look at the bus results:

In [ ]:
ac_results["bus_results"]

Notice that `Vm` now varies across buses (not all 1.0), and `Q_gen` has non-zero values.

Look at the line flows:

In [ ]:
ac_results["flow_results"]

`Q_from_to` and `Q_to_from` now show reactive power flows, and `P_from_to` differs from
`P_to_from` due to losses.

## When AC Power Flow Fails
Unlike DC power flow, AC power flow is iterative and not guaranteed to converge. Systems
with high impedance lines, poor initial voltage profiles, or insufficient reactive power
support can cause the solver to fail. When this happens, `solve_power_flow` returns
`missing`: you'll also see a logged error. If you encounter convergence failures, consider
using a more robust solver such as `TrustRegionACPowerFlow` or `RobustHomotopyPowerFlow`.